
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>


# Demo - Advanced Data Quality Checks and Expectations in Spark Declarative Pipelines

#### Duration: ~25-30 Minutes

## Overview

This demonstration showcases enterprise-grade data quality management using Spark Declarative Pipelines (Lakeflow) with advanced quality expectations and quarantine patterns. You will build a comprehensive medallion architecture that implements three critical data engineering patterns: range-based validation, automatic schema evolution, and zero-loss quarantine processing.

The demo simulates a real-world scenario where order data arrives with varying quality levels and evolving schemas. Using official Databricks patterns, you will process this data through bronze, silver, and gold layers while implementing robust quality controls that catch data anomalies, handle schema changes seamlessly, and ensure zero data loss through intelligent quarantine mechanisms.

This demonstration uses a two-run approach: Run 1 establishes a clean baseline with 157 perfect records to validate the quality framework, while Run 2 introduces 60 records containing intentional quality issues and schema evolution to demonstrate the system's resilience and monitoring capabilities.

## Learning Objectives

By the end of this demonstration, you will be able to:

- **Implement comprehensive data validation** using 6 quality expectations covering NOT NULL checks, numeric ranges, and date validations with proper constraint syntax
- **Handle automatic schema evolution** without pipeline code changes using flexible bronze layer design and schema hints
- **Build quarantine patterns** using official Databricks inverse logic to capture invalid records while maintaining zero data loss and detailed failure tracking
- **Monitor data quality metrics** and analyze expectation performance to identify specific data quality issues and develop remediation strategies

## REQUIRED - SELECT A COMPUTE ENVIRONMENT

<div style="
  border-left: 4px solid #f44336;
  background: #ffebee;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#c62828; margin-bottom:6px; font-size: 1.1em;">Select Serverless Compute</strong>
  <div style="color:#333;">

Before starting this notebook, select the required compute environment listed below.

- **Serverless Compute, Version 6**  
  - [How to select an environment version](https://docs.databricks.com/aws/en/compute/serverless/dependencies#-select-an-environment-version)
  </div>
</div>


## Demo Architecture Overview

This demo demonstrates how to build a robust data pipeline using Databricks medallion architecture, focusing on data quality enforcement and schema flexibility.

### Demo Pipeline Architecture

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1690 560" width="100%" role="img" aria-label="Advanced data quality pipeline overview: sales CSV files flow through bronze raw and clean tables into a silver table with quality checks, then valid, quarantined and analytics outputs" style="font-family:'DM Sans',-apple-system,Arial,Helvetica,sans-serif;max-width:1690px;display:block;margin:12px auto;"><defs><marker id="od-ah-g" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-reverse"><path d="M0,0 L10,5 L0,10 z" fill="#7C8B94"/></marker><marker id="od-ah-n" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-reverse"><path d="M0,0 L10,5 L0,10 z" fill="#1B3139"/></marker><marker id="od-ah-t" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-reverse"><path d="M0,0 L10,5 L0,10 z" fill="#1B5062"/></marker><filter id="od-sh" x="-8%" y="-8%" width="116%" height="130%"><feDropShadow dx="0" dy="2" stdDeviation="3" flood-color="#0B2026" flood-opacity="0.13"/></filter></defs><text x="14" y="30" font-size="20" font-weight="400" fill="#1B3139" text-anchor="start" style="font-family:'DM Sans',-apple-system,Arial,Helvetica,sans-serif" >Cloud Storage</text><rect x="10" y="42" width="168" height="400" rx="18" fill="#fff" stroke="#9AA5AB" stroke-width="1.6" /><rect x="24" y="62" width="140" height="360" rx="8" fill="none" stroke="#8C98A0" stroke-width="1.5" stroke-dasharray="5 4" /><text x="94" y="90" font-size="17" font-weight="700" fill="#0B2026" text-anchor="middle" style="font-family:'DM Sans',-apple-system,Arial,Helvetica,sans-serif" >Sales</text><text x="94" y="110" font-size="14.5" font-weight="400" fill="#0B2026" text-anchor="middle" style="font-family:'DM Sans',-apple-system,Arial,Helvetica,sans-serif" >CSV Files</text><g transform="translate(78,130) scale(1.0)"><path d="M0,0 h22 l12,12 v32 h-34 z" fill="#fff" stroke="#FF5F46" stroke-width="2" stroke-linejoin="round"/><path d="M22,0 v12 h12" fill="none" stroke="#FF5F46" stroke-width="2" stroke-linejoin="round"/><rect x="2" y="25" width="30" height="14" rx="3" fill="#FF5F46"/><text x="17" y="35.5" font-size="9.5" font-weight="700" fill="#fff" text-anchor="middle" style="font-family:'DM Sans',-apple-system,Arial,Helvetica,sans-serif" >CSV</text></g><text x="94" y="196" font-size="13" font-weight="400" fill="#0B2026" text-anchor="middle" style="font-family:'DM Sans',-apple-system,Arial,Helvetica,sans-serif" >File 1</text><g transform="translate(78,226) scale(1.0)"><path d="M0,0 h22 l12,12 v32 h-34 z" fill="#fff" stroke="#FF5F46" stroke-width="2" stroke-linejoin="round"/><path d="M22,0 v12 h12" fill="none" stroke="#FF5F46" stroke-width="2" stroke-linejoin="round"/><rect x="2" y="25" width="30" height="14" rx="3" fill="#FF5F46"/><text x="17" y="35.5" font-size="9.5" font-weight="700" fill="#fff" text-anchor="middle" style="font-family:'DM Sans',-apple-system,Arial,Helvetica,sans-serif" >CSV</text></g><text x="94" y="292" font-size="13" font-weight="400" fill="#0B2026" text-anchor="middle" style="font-family:'DM Sans',-apple-system,Arial,Helvetica,sans-serif" >File 2</text><g transform="translate(78,322) scale(1.0)"><path d="M0,0 h22 l12,12 v32 h-34 z" fill="#fff" stroke="#FF5F46" stroke-width="2" stroke-linejoin="round"/><path d="M22,0 v12 h12" fill="none" stroke="#FF5F46" stroke-width="2" stroke-linejoin="round"/><rect x="2" y="25" width="30" height="14" rx="3" fill="#FF5F46"/><text x="17" y="35.5" font-size="9.5" font-weight="700" fill="#fff" text-anchor="middle" style="font-family:'DM Sans',-apple-system,Arial,Helvetica,sans-serif" >CSV</text></g><text x="94" y="388" font-size="13" font-weight="400" fill="#0B2026" text-anchor="middle" style="font-family:'DM Sans',-apple-system,Arial,Helvetica,sans-serif" >File 3</text><path d="M166,281 H214" fill="none" stroke="#1B5062" stroke-width="2.2" marker-end="url(#od-ah-t)" stroke-linecap="round" stroke-linejoin="round"/><svg x="216" y="218" width="34" height="31.0" viewBox="0 0 56 51" fill="#1B3139"><rect x="14" y="6" width="34" height="3"/><rect x="14" y="6" width="3" height="33"/><rect x="45" y="6" width="3" height="19"/><rect x="14" y="17" width="34" height="3"/><rect x="24" y="20" width="3" height="17"/><rect x="14" y="37" width="17" height="3"/><path d="M32,29 q2.1,-4.6 4.2,0 t4.2,0 t4.2,0 t4.2,0" fill="none" stroke="#1B3139" stroke-width="2.6" stroke-linecap="round"/><path d="M32,36 q2.1,-4.6 4.2,0 t4.2,0 t4.2,0 t4.2,0" fill="none" stroke="#1B3139" stroke-width="2.6" stroke-linecap="round"/></svg><text x="258" y="244" font-size="14.5" font-weight="700" fill="#0B2026" text-anchor="start" style="font-family:'DM Sans',-apple-system,Arial,Helvetica,sans-serif" >Streaming Table</text><rect x="216" y="258" width="214" height="46" rx="8" fill="#FF5F46" stroke="none" stroke-width="0" filter="url(#od-sh)"/><text x="323.0" y="286" font-size="12.5" font-weight="700" fill="#fff" text-anchor="middle" style="font-family:'DM Sans',-apple-system,Arial,Helvetica,sans-serif" >sales_bronze_raw_demo</text><path d="M432,281 H468" fill="none" stroke="#1B5062" stroke-width="2.2" marker-end="url(#od-ah-t)" stroke-linecap="round" stroke-linejoin="round"/><svg x="470" y="218" width="34" height="31.0" viewBox="0 0 56 51" fill="#1B3139"><rect x="14" y="6" width="34" height="3"/><rect x="14" y="6" width="3" height="33"/><rect x="45" y="6" width="3" height="19"/><rect x="14" y="17" width="34" height="3"/><rect x="24" y="20" width="3" height="17"/><rect x="14" y="37" width="17" height="3"/><path d="M32,29 q2.1,-4.6 4.2,0 t4.2,0 t4.2,0 t4.2,0" fill="none" stroke="#1B3139" stroke-width="2.6" stroke-linecap="round"/><path d="M32,36 q2.1,-4.6 4.2,0 t4.2,0 t4.2,0 t4.2,0" fill="none" stroke="#1B3139" stroke-width="2.6" stroke-linecap="round"/></svg><text x="512" y="244" font-size="14.5" font-weight="700" fill="#0B2026" text-anchor="start" style="font-family:'DM Sans',-apple-system,Arial,Helvetica,sans-serif" >Streaming Table</text><rect x="470" y="258" width="226" height="46" rx="8" fill="#FF5F46" stroke="none" stroke-width="0" filter="url(#od-sh)"/><text x="583.0" y="286" font-size="12.5" font-weight="700" fill="#fff" text-anchor="middle" style="font-family:'DM Sans',-apple-system,Arial,Helvetica,sans-serif" >sales_bronze_clean_demo</text><path d="M698,281 H736" fill="none" stroke="#1B5062" stroke-width="2.2" marker-end="url(#od-ah-t)" stroke-linecap="round" stroke-linejoin="round"/><text x="928" y="150" font-size="22" font-weight="700" fill="#0B2026" text-anchor="middle" style="font-family:'DM Sans',-apple-system,Arial,Helvetica,sans-serif" >Applying Data</text><text x="928" y="176" font-size="22" font-weight="700" fill="#0B2026" text-anchor="middle" style="font-family:'DM Sans',-apple-system,Arial,Helvetica,sans-serif" >Quality Checks</text><rect x="738" y="190" width="380" height="224" rx="0" fill="#F6F4F1" stroke="#8C98A0" stroke-width="1.2" /><svg x="752" y="218" width="34" height="31.0" viewBox="0 0 56 51" fill="#1B3139"><rect x="14" y="6" width="34" height="3"/><rect x="14" y="6" width="3" height="33"/><rect x="45" y="6" width="3" height="19"/><rect x="14" y="17" width="34" height="3"/><rect x="24" y="20" width="3" height="17"/><rect x="14" y="37" width="17" height="3"/><path d="M32,29 q2.1,-4.6 4.2,0 t4.2,0 t4.2,0 t4.2,0" fill="none" stroke="#1B3139" stroke-width="2.6" stroke-linecap="round"/><path d="M32,36 q2.1,-4.6 4.2,0 t4.2,0 t4.2,0 t4.2,0" fill="none" stroke="#1B3139" stroke-width="2.6" stroke-linecap="round"/></svg><text x="794" y="244" font-size="14.5" font-weight="700" fill="#0B2026" text-anchor="start" style="font-family:'DM Sans',-apple-system,Arial,Helvetica,sans-serif" >Streaming Table</text><rect x="752" y="258" width="352" height="46" rx="8" fill="#FF5F46" stroke="none" stroke-width="0" filter="url(#od-sh)"/><text x="928.0" y="286" font-size="14" font-weight="700" fill="#fff" text-anchor="middle" style="font-family:'DM Sans',-apple-system,Arial,Helvetica,sans-serif" >sales_silver_dq_demo</text><text x="928" y="352" font-size="15.5" text-anchor="middle" fill="#1B3139" style="font-family:'DM Sans',-apple-system,Arial,Helvetica,sans-serif">Applying Data Quality Checks</text><text x="928" y="374" font-size="15.5" text-anchor="middle" fill="#1B3139" style="font-family:'DM Sans',-apple-system,Arial,Helvetica,sans-serif">and Writing Quarantine logic</text><svg x="1150" y="54" width="34" height="31.0" viewBox="0 0 56 51" fill="#1B3139"><rect x="14" y="6" width="34" height="3"/><rect x="14" y="6" width="3" height="33"/><rect x="45" y="6" width="3" height="19"/><rect x="14" y="17" width="34" height="3"/><rect x="24" y="20" width="3" height="17"/><rect x="14" y="37" width="17" height="3"/><path d="M32,29 q2.1,-4.6 4.2,0 t4.2,0 t4.2,0 t4.2,0" fill="none" stroke="#1B3139" stroke-width="2.6" stroke-linecap="round"/><path d="M32,36 q2.1,-4.6 4.2,0 t4.2,0 t4.2,0 t4.2,0" fill="none" stroke="#1B3139" stroke-width="2.6" stroke-linecap="round"/></svg><text x="1192" y="80" font-size="14.5" font-weight="700" fill="#0B2026" text-anchor="start" style="font-family:'DM Sans',-apple-system,Arial,Helvetica,sans-serif" >Streaming Table</text><rect x="1150" y="94" width="232" height="46" rx="8" fill="#FF5F46" stroke="none" stroke-width="0" filter="url(#od-sh)"/><text x="1266.0" y="122" font-size="12.5" font-weight="700" fill="#fff" text-anchor="middle" style="font-family:'DM Sans',-apple-system,Arial,Helvetica,sans-serif" >sales_silver_valid_demo</text><path d="M1120,281 H1126 V125 Q1126,117 1134,117 H1145" fill="none" stroke="#1B5062" stroke-width="2.2" marker-end="url(#od-ah-t)" stroke-linecap="round" stroke-linejoin="round"/><svg x="1150" y="324" width="34" height="31.0" viewBox="0 0 56 51" fill="#1B3139"><rect x="14" y="6" width="34" height="3"/><rect x="14" y="6" width="3" height="33"/><rect x="45" y="6" width="3" height="19"/><rect x="14" y="17" width="34" height="3"/><rect x="24" y="20" width="3" height="17"/><rect x="14" y="37" width="17" height="3"/><path d="M32,29 q2.1,-4.6 4.2,0 t4.2,0 t4.2,0 t4.2,0" fill="none" stroke="#1B3139" stroke-width="2.6" stroke-linecap="round"/><path d="M32,36 q2.1,-4.6 4.2,0 t4.2,0 t4.2,0 t4.2,0" fill="none" stroke="#1B3139" stroke-width="2.6" stroke-linecap="round"/></svg><text x="1192" y="350" font-size="14.5" font-weight="700" fill="#0B2026" text-anchor="start" style="font-family:'DM Sans',-apple-system,Arial,Helvetica,sans-serif" >Streaming Table</text><rect x="1150" y="364" width="264" height="46" rx="8" fill="#FF5F46" stroke="none" stroke-width="0" filter="url(#od-sh)"/><text x="1282.0" y="392" font-size="12.5" font-weight="700" fill="#fff" text-anchor="middle" style="font-family:'DM Sans',-apple-system,Arial,Helvetica,sans-serif" >sales_silver_quarantined_demo</text><path d="M1120,281 H1126 V379 Q1126,387 1134,387 H1145" fill="none" stroke="#1B5062" stroke-width="2.2" marker-end="url(#od-ah-t)" stroke-linecap="round" stroke-linejoin="round"/><svg x="1448" y="54" width="34" height="34" viewBox="0 0 56 51" fill="#1B3139"><rect x="14" y="6" width="34" height="3"/><rect x="14" y="6" width="3" height="33"/><rect x="45" y="6" width="3" height="19"/><rect x="14" y="17" width="34" height="3"/><rect x="24" y="20" width="3" height="17"/><rect x="14" y="37" width="17" height="3"/><rect x="32.015" y="26.436" width="10.5" height="3.5" transform="rotate(131 37.265 28.186)"/><rect x="32.5" y="31" width="15" height="3"/><rect x="37.485" y="35.064" width="10.5" height="3.5" transform="rotate(131 42.735 36.814)"/></svg><text x="1490" y="80" font-size="14.5" font-weight="700" fill="#0B2026" text-anchor="start" style="font-family:'DM Sans',-apple-system,Arial,Helvetica,sans-serif" >Materialized View</text><rect x="1448" y="94" width="214" height="46" rx="8" fill="#FF5F46" stroke="none" stroke-width="0" filter="url(#od-sh)"/><text x="1555.0" y="122" font-size="12.5" font-weight="700" fill="#fff" text-anchor="middle" style="font-family:'DM Sans',-apple-system,Arial,Helvetica,sans-serif" >sales_analytics_demo</text><path d="M1384,117 H1444" fill="none" stroke="#1B5062" stroke-width="2.2" marker-end="url(#od-ah-t)" stroke-linecap="round" stroke-linejoin="round"/><path d="M216,452 H1064 V434 L1146,478 L1064,522 V504 H216 z" fill="#1B5062"/><text x="640" y="486" font-size="18" text-anchor="middle" fill="#fff" style="font-family:'DM Sans',-apple-system,Arial,Helvetica,sans-serif"><tspan font-weight="700">Incrementally</tspan> process <tspan font-weight="700">new data</tspan> in the streaming tables (ST)</text></svg>

### Demo Execution Flow

- **Run 1:** Clean baseline data (157 records) — all pass quality checks, establishes 100% quality baseline
- **Run 2:** Data with quality issues (60 records, 2 new columns) — demonstrates issue detection and schema evolution

## A. Setup

> **Workspace setup:** Using a Databricks Academy (Vocareum) workspace? Your catalog is created for you, so you can start. Using your own workspace or Databricks Free Edition? Complete **Course Overview → B. Workspace Setup** first (catalog creation and permissions, Marketplace data). Run this notebook only in a development or sandbox workspace.

### A1. Configure Your Catalog and Schema

Run the cell below to initialize your environment. This setup step performs the following actions:

- **Creates or validates catalog access** when running outside of a Databricks provided Vocareum workspace
- **Creates three schemas** in your specified catalog:
    - **dq_1_bronze** - Raw data ingestion layer
    - **dq_2_silver** - Validated and transformed data layer  
    - **dq_3_gold** - Analytics-ready aggregated data layer
- **Creates volume storage** for source data processing:
    - `sales` volume: Active processing location for pipeline consumption
    - `ops` volume: Staging location containing multiple data files
- **Verifies compute environment** compatibility with Lakeflow pipelines

This ensures that all schemas, tables, and objects are created in your designated catalog with proper isolation.

**Important:** You must have permission to create catalogs in your own non-Vocareum workspace. If you do not have the required permissions, this step will fail. If you cannot create catalogs, see **Course Overview → B2. Troubleshooting: Missing Create Catalog Permissions** before continuing.

In [ ]:
%run ./Includes/Classroom-Setup-dq

### A2. Verify Volume Path Configuration

Run the cell below to view the value of the `my_vol_path` variable and confirm that it references your **your-catalog.dq_1_bronze** path. This path will be used to dynamically reference your source volumes throughout this demonstration.

In [ ]:
print(my_vol_path)

## B. Explore Source Data Characteristics

Before building the pipeline, examine the source data to understand its structure, quality characteristics, and schema evolution patterns. This analysis will inform our quality expectation design.

### B1. Examine the Source Files

The **ops** volume holds several files that are moved incrementally into the **sales** volume, which the pipeline reads. This simulates real-world data arrival.

Run the cell below to list both volumes. Notice that **ops** contains multiple files, while **sales** currently contains only one.

In [ ]:
# List the files in the ops and sales volumes
for volume in ['ops', 'sales']:
    print(f'--- {volume}')
    display(spark.sql(f"LIST '{my_vol_path}/{volume}'"))

### B2. Compare the First Two Files

| | `sales_1.csv` | `sales_2.csv` |
|---|---|---|
| **Records** | 157 | 60 |
| **Columns** | 16 | 18 (the 16 original columns plus the new `order_status` and `shipping_cost`) |
| **Quality** | Clean: no missing business keys, discount percentages between 0 and 100, recent order dates | Intentional quality issues (below) |
| **Purpose** | Quality baseline for the first run | Schema evolution and quality issues for the second run |

**Intentional quality issues in `sales_2.csv`:**
- Invalid discount percentages (for example 120, outside the 0-100 range)
- Historical order dates (for example 1930-12-31)
- Missing `customer_id` values
- Invalid shipping costs (above 100 or below 0)

About 85% of the 60 records are expected to pass all checks, and about 15% to be quarantined with the failure reasons.

Run the cell below to preview both files. The previews also show a `_rescued_data` column that `read_files` adds, which is not counted in the column numbers above.

In [ ]:
# Preview the first file (clean baseline) and the second file (quality issues + new columns)
for file in ['sales_1.csv', 'sales_2.csv']:
    print(f'--- {file}')
    display(spark.sql(f"SELECT * FROM read_files('{my_vol_path}/ops/{file}')"))

## C. Create the Spark Declarative Pipeline

Now that we understand the data characteristics and quality challenges, create the Spark Declarative Pipeline with comprehensive data quality expectations and quarantine handling.

### C1. Create a Apache Spark™ Declarative Pipeline

Complete the following steps to create your Spark Declarative Pipeline using the Lakeflow Pipelines Editor:

1. In the main navigation pane, right-click **Jobs & Pipelines** and select **Open link in New Tab**.  

2. In the new tab, select **Create → ETL Pipeline**.  

3. Configure the pipeline settings:
   - **Pipeline Name**: `demo_dq_yourname`
   - **Default Catalog**: `YOUR_LABUSER_CATALOG`
   - **Default Schema**: `dq_1_bronze`  
   **NOTE:** Clear the selected schema using the cross icon to view all schemas.

4. **Rename pipeline components**:
   - Rename the **transformations** folder to `dq_pipeline`
   - Rename **my_transformation.py** file to `bronze_ingestion.sql`

5. Leave the **Lakeflow Pipelines Editor** page open for the next steps.

### C2. Configure Pipeline Parameters

1. Run the cell below to retrieve the key-value pairs needed to set your pipeline configuration parameters for the **source volume**.

In [ ]:
config_parameters = [
    ('source', f'{my_vol_path}/sales')
]

for key, value in config_parameters:
    print(f"Key: {key}\nValue: {value}\n")

2. Add the parameter to your **Spark Declarative Pipeline**: open **Settings**, under **Configuration** select **Add configuration**, enter the key and value shown above, and select **Save**. This lets your pipeline reference the volume through a parameter.

**NOTE:** For more details on configuration parameters, see the Databricks documentation: [Use parameters with Spark Declarative Pipelines](https://docs.databricks.com/aws/en/ldp/parameters)

## D. Create Bronze Layer - Raw Ingestion with Schema Evolution

The bronze layer implements schema evolution by using the `read_files` function with flexible column handling and schema hints for future column compatibility.

### D1. Create Bronze Table with Schema Evolution Support

Copy the code below into your `bronze_ingestion.sql` file. This implementation uses a STRING-based approach for maximum schema flexibility and includes schema hints for future columns.

<button onclick="copyBlock()">Copy to clipboard</button>
<pre id="copy-block" style="font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, 'Liberation Mono', 'Courier New', monospace; border:1px solid #e5e7eb; border-radius:10px; background:#f8fafc; padding:14px 16px; font-size:0.85rem; line-height:1.35; white-space:pre;">
<code>

------------------------------------------
-- BRONZE LAYER: RAW DATA INGESTION
------------------------------------------
CREATE OR REFRESH STREAMING TABLE dq_1_bronze.sales_bronze_raw_demo
AS
SELECT 
  -- Core business fields (all as STRING for schema flexibility)
  CAST(subsidiary_id AS STRING) AS subsidiary_id,
  CAST(order_id AS STRING) AS order_id,
  CAST(order_timestamp AS STRING) AS order_timestamp,
  CAST(customer_id AS STRING) AS customer_id,
  CAST(region AS STRING) AS region,
  CAST(country AS STRING) AS country,
  CAST(city AS STRING) AS city,
  CAST(channel AS STRING) AS channel,
  CAST(sku AS STRING) AS sku,
  CAST(category AS STRING) AS category,
  CAST(qty AS STRING) AS qty,
  CAST(unit_price AS STRING) AS unit_price,
  CAST(discount_pct AS STRING) AS discount_pct,
  CAST(coupon_code AS STRING) AS coupon_code,
  CAST(total_amount AS STRING) AS total_amount,
  CAST(order_date AS STRING) AS order_date,

  -- New columns for schema evolution (will be null initially)
  CAST(order_status AS STRING) AS order_status,
  CAST(shipping_cost AS STRING) AS shipping_cost,

  -- Rescued data column for parsing issues
  CAST(_rescued_data AS STRING) AS _rescued_data,

  -- Metadata columns for lineage tracking
  _metadata.file_name AS source_file,
  _metadata.file_modification_time AS file_mod_time
FROM STREAM read_files(
  '${source}',
  format => 'csv',
  schemaHints => 'order_status STRING, shipping_cost STRING'
);
</code></pre>

<script>
function copyBlock() {
  const el = document.getElementById("copy-block");
  if (!el) return;

  const text = el.innerText;

  // Preferred modern API
  if (navigator.clipboard && navigator.clipboard.writeText) {
    navigator.clipboard.writeText(text)
      .then(() => alert("Copied to clipboard"))
      .catch(err => {
        console.error("Clipboard write failed:", err);
        fallbackCopy(text);
      });
  } else {
    fallbackCopy(text);
  }
}

function fallbackCopy(text) {
  const textarea = document.createElement("textarea");
  textarea.value = text;
  textarea.style.position = "fixed";
  textarea.style.left = "-9999px";
  document.body.appendChild(textarea);
  textarea.select();
  try {
    document.execCommand("copy");
    alert("Copied to clipboard");
  } catch (err) {
    console.error("Fallback copy failed:", err);
    alert("Could not copy to clipboard. Please copy manually.");
  } finally {
    document.body.removeChild(textarea);
  }
}
</script>

**Review the Bronze Layer Design Patterns:**

**Schema Evolution Strategy:**
- **STRING storage**: All business columns stored as STRING for maximum flexibility and compatibility
- **Pre-defined schema**: Includes both original and two future columns for seamless evolution
- **Schema hints**: Declares expected new columns (`order_status`, `shipping_cost`) even when not yet present in data
- **Null handling**: New columns will be NULL for existing files, populated for future files

**Data Lineage and Protection:**
- **Auto Loader**: `read_files` function enables incremental processing with checkpoint management
- **Metadata capture**: Source file names and modification timestamps for complete data lineage
- **Rescued data**: `_rescued_data` column captures any parsing issues for investigation and recovery
- **Parameter reference**: `${source}` enables dynamic path configuration

**Technical Benefits:**
- No pipeline code changes required when new columns arrive
- Backward compatibility with existing data files
- Forward compatibility with evolving business requirements

For more information about schema hints functionality, see the [Databricks documentation on schema hints](https://docs.databricks.com/aws/en/ingestion/cloud-object-storage/auto-loader/schema#override-schema-inference-with-schema-hints).

### D2. Create Clean Bronze Table

Create a streaming intermediate Bronze table that performs safe type casting from the raw ingested data, without applying any quality checks. This step ensures clean type conversion before data validation.

- Creates a clean streaming Bronze table from the source data stream.
- Uses `TRY_CAST` to safely convert data types (dates, numbers) without failing on bad data.
- Passes through all columns without applying any data quality filters yet.

<button onclick="copyBlock()">Copy to clipboard</button>

<pre id="copy-block" style="font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, 'Liberation Mono', 'Courier New', monospace; border:1px solid #e5e7eb; border-radius:10px; background:#f8fafc; padding:14px 16px; font-size:0.85rem; line-height:1.35; white-space:pre;">
<code>
--========================================================================
-- BRONZE LAYER: INTERMEDIATE TRANSFORMATION TABLE
-- Purpose: Clean type casting from Bronze Raw (no quality checks yet)
--========================================================================

CREATE OR REFRESH STREAMING TABLE dq_1_bronze.sales_bronze_clean_demo
COMMENT "Intermediate table - type casting only, no quality checks"
AS
SELECT
  -- Direct pass-through and safe type casting
  subsidiary_id,
  order_id,
  TRY_CAST(order_timestamp AS TIMESTAMP) AS order_timestamp,
  TRY_CAST(order_date AS DATE) AS order_date,
  customer_id,
  region,
  country,
  city,
  channel,
  sku,
  category,
  TRY_CAST(qty AS INT) AS qty,
  TRY_CAST(unit_price AS DOUBLE) AS unit_price,
  TRY_CAST(discount_pct AS DOUBLE) AS discount_pct,
  TRY_CAST(total_amount AS DOUBLE) AS total_amount,
  coupon_code,
  order_status,
  TRY_CAST(shipping_cost AS DOUBLE) AS shipping_cost,
  source_file

FROM STREAM dq_1_bronze.sales_bronze_raw_demo;
</code></pre>

<script>
function copyBlock() {
  const el = document.getElementById("copy-block");
  if (!el) return;

  const text = el.innerText;

  // Preferred modern API
  if (navigator.clipboard && navigator.clipboard.writeText) {
    navigator.clipboard.writeText(text)
      .then(() => alert("Copied to clipboard"))
      .catch(err => {
        console.error("Clipboard write failed:", err);
        fallbackCopy(text);
      });
  } else {
    fallbackCopy(text);
  }
}

function fallbackCopy(text) {
  const textarea = document.createElement("textarea");
  textarea.value = text;
  textarea.style.position = "fixed";
  textarea.style.left = "-9999px";
  document.body.appendChild(textarea);
  textarea.select();
  try {
    document.execCommand("copy");
    alert("Copied to clipboard");
  } finally {
    document.body.removeChild(textarea);
  }
}
</script>

### D3. Run and Validate the Bronze Layer

1. **Run the pipeline** and confirm that it completes successfully with no errors.

2. **Validate ingestion results** in the Lakeflow Pipelines editor:
   - Verify that **157 records** were ingested into the sales_bronze_raw_demo table from the first file
   - Select the **sales_bronze_raw_demo** table and open the **Data** tab to preview the ingested sales data
   - Confirm that new columns (`order_status`, `shipping_cost`) exist but contain NULL values

**Troubleshooting:** If your pipeline fails, verify that:
- Volume paths are correctly configured in pipeline parameters
- The `source` configuration parameter matches your volume path
- Serverless compute is selected and running

**Expected Checkpoint Results:**

<img src="https://files.training.databricks.com/binder/prod_main/advanced-techniques-with-apache-spark-declarative-pipelines-en_us-1.1.0/images/20261029T211417Z/Advanced Techniques with Apache Spark Declarative Pipelines/Includes/images/dq/checkpoint_bronze.png" alt="Bronze Layer Results" width="1200">

### D4. Query and Analyze Bronze Layer

Run these queries to validate the bronze layer tables:

- Confirm that all records are ingested into the raw bronze table
- Check that `schema_hints` worked successfully

In [ ]:
%sql
-- Verify record ingestion by source file (should show only sales_1.csv)
SELECT 
  source_file,
  COUNT(*) AS record_count,
  'Clean baseline data' AS data_profile
FROM dq_1_bronze.sales_bronze_raw_demo
GROUP BY source_file
ORDER BY source_file;

In [ ]:
%sql
-- Verify schema evolution readiness
-- Total Column should be 21
DESCRIBE TABLE dq_1_bronze.sales_bronze_raw_demo;

## E. Create Silver Layer with Data Quality Expectations and Quarantine Logic

- In this layer, we will first type-cast columns to their appropriate data formats.
- Next, we will apply data quality expectations.
- Finally, based on these expectations, we will separate the data into two tables: one for clean records and one for invalid records, using inverse logic.

### E1. Understanding the Data Quality Expectations

Before creating the table with expectations, review what we check and why. You already know the `CONSTRAINT ... EXPECT ... ON VIOLATION` syntax and the WARN, DROP ROW and FAIL UPDATE actions.

**Our strategy:** use **WARN** (the default) for all six expectations, so the pipeline keeps running and every record is retained for quarantine analysis.

**Our 6 expectations, in three categories:**
- **Business keys (critical for data integrity):** `check_subsidiary_id`, `check_customer_id` and `check_sku` must be NOT NULL, so every order can be attributed to a subsidiary, a customer and a product.
- **Range-based business rules:** `valid_discount_range` (discount between 0% and 100%, which catches negative and impossible values) and `valid_date_range` (order date within the last 4 years, which catches archived data and future dates).
  - **NOTE:** Since our data is static, `valid_date_range` is checked against **01-01-2026**.
- **Schema evolution:** `valid_shipping_cost` (shipping cost between $0 and $100) applies to a column that is not in the first file. Once the column appears, the expectation enforces it automatically. For details, see [Schema Evolution Pattern](https://docs.databricks.com/aws/en/ldp/expectation-patterns?language=SQL#schema-evolution-pattern).

### E2. Understanding the Quarantine Records Pattern

You saw this pattern in the lecture. Records that violate expectations are routed to a separate quarantine table for analysis and remediation instead of being dropped, so **no data is lost**. The primary table keeps every record and uses WARN expectations to log violations, inverse logic (a NOT of all the rules) flags the failed records, and valid and invalid records then flow to separate tables.

> **Learn More:**  
> For more on quarantine frameworks and best practices, see the [Databricks documentation on expectation patterns and quarantining invalid records](https://docs.databricks.com/aws/en/ldp/expectation-patterns?language=Python%20Module#quarantine-invalid-records).

### E3. Create a New SQL File in your Pipeline

In your `dq_pipeline` folder, open the kebab menu, select **Create file**, choose **SQL**, and name it `silver_transformation.sql`.

### E4. Create Quarantine Table with Quality Expectations


Add the following code to your `silver_transformation.sql` file. This creates the main quarantine table with all 6 data quality expectations defined.

<button onclick="copyBlock()">Copy to clipboard</button>
<pre id="copy-block" style="font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, 'Liberation Mono', 'Courier New', monospace; border:1px solid #e5e7eb; border-radius:10px; background:#f8fafc; padding:14px 16px; font-size:0.85rem; line-height:1.35; white-space:pre;">
<code>
--========================================================================
-- SILVER LAYER - STEP 1: QUARANTINE TABLE WITH EXPECTATIONS
-- Purpose: Define table structure and 6 data quality expectations
--========================================================================

CREATE OR REFRESH STREAMING TABLE dq_2_silver.sales_silver_dq_demo
(
  -- Business columns with proper data types
  subsidiary_id STRING,
  order_id STRING,
  order_timestamp TIMESTAMP,
  order_date DATE,
  customer_id STRING,
  region STRING,
  country STRING,
  city STRING,
  channel STRING,
  sku STRING,
  category STRING,
  qty INT,
  unit_price DOUBLE,
  discount_pct DOUBLE,
  total_amount DOUBLE,
  coupon_code STRING,
  order_status STRING,
  shipping_cost DOUBLE,
  source_file STRING,

  -- Quality tracking columns for quarantine analysis
  is_quarantined BOOLEAN,
  quarantine_reason STRING,

  --========================================================================
  -- 6 DATA QUALITY EXPECTATIONS (WARN ACTION FOR QUARANTINE PATTERN)
  --========================================================================

  CONSTRAINT check_subsidiary_id 
    EXPECT (subsidiary_id IS NOT NULL),

  CONSTRAINT check_customer_id 
    EXPECT (customer_id IS NOT NULL),

  CONSTRAINT check_sku 
    EXPECT (sku IS NOT NULL),

  CONSTRAINT valid_discount_range
    EXPECT (discount_pct IS NULL OR (discount_pct >= 0 AND discount_pct <= 100)),

  -- Since the data is static, we check that order_date falls between 01-01-2026 and 4 years prior, instead of using the current date.
  CONSTRAINT valid_date_range 
    EXPECT (order_date IS NULL OR 
        (order_date >= DATE_SUB(DATE '2026-01-01', 1460) AND 
          order_date <= DATE '2026-01-01')),

  CONSTRAINT valid_shipping_cost EXPECT (
    CASE WHEN shipping_cost IS NOT NULL THEN shipping_cost > 0 AND shipping_cost < 100 ELSE TRUE END
  )
)
COMMENT "Quarantine table with 6 expectations - supports inverse logic pattern"
TBLPROPERTIES (
  'quality.layer' = 'silver_quarantine',
  'quality.pattern' = 'inverse_logic'
)
PARTITIONED BY (is_quarantined);
</code></pre>

<script>
function copyBlock() {
  const el = document.getElementById("copy-block");
  if (!el) return;

  const text = el.innerText;

  // Preferred modern API
  if (navigator.clipboard && navigator.clipboard.writeText) {
    navigator.clipboard.writeText(text)
      .then(() => alert("Copied to clipboard"))
      .catch(err => {
        console.error("Clipboard write failed:", err);
        fallbackCopy(text);
      });
  } else {
    fallbackCopy(text);
  }
}

function fallbackCopy(text) {
  const textarea = document.createElement("textarea");
  textarea.value = text;
  textarea.style.position = "fixed";
  textarea.style.left = "-9999px";
  document.body.appendChild(textarea);
  textarea.select();
  try {
    document.execCommand("copy");
    alert("Copied to clipboard");
  } finally {
    document.body.removeChild(textarea);
  }
}
</script>

**Review the Expectation Framework:**

**Expectations Summary:**

| # | Expectation Name | Validates | Range/Rule | Violation Example |
|---|-----------------|-----------|------------|---------------------|
| 1 | check_subsidiary_id | Business Key | NOT NULL | Missing subsidiary IDs |
| 2 | check_customer_id | Business Key | NOT NULL | Missing customer IDs |
| 3 | check_sku | Business Key | NOT NULL | Missing SKUs |
| 4 | valid_discount_range | Discount | 0%-100% | discount = -10.73% or 120% |
| 5 | valid_date_range | Date | Last 4 years | date = 1930-12-31 |
| 6 | valid_shipping_cost | Shipping | $0-$100 | shipping_cost = -5 or 150 |

**Quality Tracking Design:**
- **is_quarantined**: Boolean flag indicating whether record failed any expectation
- **quarantine_reason**: Detailed string listing all failed validations for remediation
- **Partitioning**: Separates quarantined and valid data for performance optimization

**Table Properties:**
- Metadata tags identify this as a quarantine pattern implementation
- Enables monitoring and governance tools to recognize the quality framework

### E5. Create Inverse Logic Flow for Quarantine Population

Add the inverse logic flow to populate the quarantine table with quality tracking information. This flow implements the official Databricks quarantine pattern by using inverse logic to identify and flag records that fail any quality expectations.

<button onclick="copyBlock()">Copy to clipboard</button>

<pre id="copy-block" style="font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, 'Liberation Mono', 'Courier New', monospace; border:1px solid #e5e7eb; border-radius:10px; background:#f8fafc; padding:14px 16px; font-size:0.85rem; line-height:1.35; white-space:pre;">
<code>
--========================================================================
-- SILVER LAYER - STEP 2: FLOW TO APPLY INVERSE LOGIC
-- Purpose: Calculate is_quarantined flag and populate quarantine table
--========================================================================

CREATE FLOW apply_inverse_logic_flow
AS
INSERT INTO dq_2_silver.sales_silver_dq_demo BY NAME
SELECT
  -- Pass through all business columns
  subsidiary_id,
  order_id,
  order_timestamp,
  order_date,
  customer_id,
  region,
  country,
  city,
  channel,
  sku,
  category,
  qty,
  unit_price,
  discount_pct,
  total_amount,
  coupon_code,
  order_status,
  shipping_cost,
  source_file,

  --========================================================================
  -- INVERSE LOGIC: Mark as quarantined if ANY expectation fails
  --========================================================================
  NOT (
    (subsidiary_id IS NOT NULL) AND
    (customer_id IS NOT NULL) AND
    (sku IS NOT NULL) AND
    (discount_pct IS NULL OR (discount_pct >= 0 AND discount_pct <= 100)) AND
    (order_date IS NULL OR 
     (order_date >= DATE_SUB(DATE '2026-01-01', 1460) AND 
      order_date <= DATE '2026-01-01')) AND
    (shipping_cost IS NULL OR (shipping_cost > 0 AND shipping_cost < 100))
  ) AS is_quarantined,

  --========================================================================
  -- BUILD DETAILED QUARANTINE REASON STRING
  --========================================================================
  CONCAT_WS('; ',
    CASE WHEN subsidiary_id IS NULL 
         THEN 'Missing subsidiary_id' END,
    CASE WHEN customer_id IS NULL 
         THEN 'Missing customer_id' END,
    CASE WHEN sku IS NULL 
         THEN 'Missing sku' END,
    CASE WHEN discount_pct IS NOT NULL AND (discount_pct < 0 OR discount_pct > 100) 
         THEN 'Invalid discount_pct (must be 0-100)' END,
    CASE WHEN order_date IS NOT NULL AND 
              (order_date < DATE_SUB(DATE '2026-01-01', 1460) OR order_date > DATE '2026-01-01') 
         THEN 'Invalid order_date (outside 4-year range)' END,
    CASE WHEN shipping_cost IS NOT NULL AND (shipping_cost <= 0 OR shipping_cost >= 100)
         THEN 'Invalid shipping_cost (must be 0-100)' END
  ) AS quarantine_reason

FROM STREAM dq_1_bronze.sales_bronze_clean_demo;
</code></pre>

<script>
function copyBlock() {
  const el = document.getElementById("copy-block");
  if (!el) return;

  const text = el.innerText;

  // Preferred modern API
  if (navigator.clipboard && navigator.clipboard.writeText) {
    navigator.clipboard.writeText(text)
      .then(() => alert("Copied to clipboard"))
      .catch(err => {
        console.error("Clipboard write failed:", err);
        fallbackCopy(text);
      });
  } else {
    fallbackCopy(text);
  }
}

function fallbackCopy(text) {
  const textarea = document.createElement("textarea");
  textarea.value = text;
  textarea.style.position = "fixed";
  textarea.style.left = "-9999px";
  document.body.appendChild(textarea);
  textarea.select();
  try {
    document.execCommand("copy");
    alert("Copied to clipboard");
  } finally {
    document.body.removeChild(textarea);
  }
}
</script>

**Review the Inverse Logic Implementation:**

**Quarantine Flag Logic:**
- **NOT()** wrapper around all expectations combined with AND logic
- If ANY expectation fails, the record is marked `is_quarantined = TRUE`
- If ALL expectations pass, the record is marked `is_quarantined = FALSE`

**Detailed Failure Tracking:**
- **CONCAT_WS()** builds semicolon-separated failure reasons
- Each CASE statement checks for specific violation types
- Provides actionable information for data remediation teams
- Empty string for records that pass all validations

**Streaming Flow Benefits:**
- Processes records continuously as they arrive
- Maintains low latency for real-time quality monitoring
- Enables immediate quarantine identification for operational workflows

### E6. Create Separate Tables for Valid and Quarantined Records

Create separate tables for valid and quarantined records to enable zero data loss and parallel processing workflows. Add this code to your `silver_transformation.sql` file.

<button onclick="copyBlock()">Copy to clipboard</button>

<pre id="copy-block" style="font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, 'Liberation Mono', 'Courier New', monospace; border:1px solid #e5e7eb; border-radius:10px; background:#f8fafc; padding:14px 16px; font-size:0.85rem; line-height:1.35; white-space:pre;">
<code>
--========================================================================
-- SILVER LAYER - STEP 3: SEPARATE VALID AND INVALID DATA PATHS
--========================================================================

--========================================================================
-- VALID RECORDS PATH - Clean data for analytics
--========================================================================

CREATE OR REFRESH STREAMING TABLE dq_2_silver.sales_silver_valid_demo
COMMENT "Clean records passing all 6 quality checks - ready for analytics"
AS
SELECT * EXCEPT (is_quarantined, quarantine_reason)
FROM STREAM dq_2_silver.sales_silver_dq_demo
WHERE is_quarantined = FALSE;

--========================================================================
-- QUARANTINED RECORDS PATH - Invalid data for remediation
--========================================================================

CREATE OR REFRESH STREAMING TABLE dq_2_silver.sales_silver_quarantined_demo
COMMENT "Invalid records with quality violations - requires remediation"
AS
SELECT *
FROM STREAM dq_2_silver.sales_silver_dq_demo
WHERE is_quarantined = TRUE;

</code></pre>

<script>
function copyBlock() {
  const el = document.getElementById("copy-block");
  if (!el) return;

  const text = el.innerText;

  // Preferred modern API
  if (navigator.clipboard && navigator.clipboard.writeText) {
    navigator.clipboard.writeText(text)
      .then(() => alert("Copied to clipboard"))
      .catch(err => {
        console.error("Clipboard write failed:", err);
        fallbackCopy(text);
      });
  } else {
    fallbackCopy(text);
  }
}

function fallbackCopy(text) {
  const textarea = document.createElement("textarea");
  textarea.value = text;
  textarea.style.position = "fixed";
  textarea.style.left = "-9999px";
  document.body.appendChild(textarea);
  textarea.select();
  try {
    document.execCommand("copy");
    alert("Copied to clipboard");
  } finally {
    document.body.removeChild(textarea);
  }
}
</script>

**Review the code:**
- The source table keeps every record, because its WARN expectations only log violations.
- **Valid path:** `sales_silver_valid_demo` holds the records where `is_quarantined = FALSE`. **Quarantine path:** `sales_silver_quarantined_demo` holds the records where `is_quarantined = TRUE`.
- Total records = valid records + quarantined records, so no data is lost. Analytics reads only the valid table, and remediation works from the quarantined table.

## F. Create Gold Layer - Production-Ready Analytics Data

The gold layer contains pre-aggregated analytics from validated silver data, providing fast query performance for business intelligence and reporting applications.

### F1. Create Gold Layer Analytics Table

In your `dq_pipeline` folder, create a new SQL file named `gold_analytics.sql` (kebab menu → **Create file** → **SQL**) and paste the code below.

<button onclick="copyBlock()">Copy to clipboard</button>

<pre id="copy-block" style="font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, 'Liberation Mono', 'Courier New', monospace; border:1px solid #e5e7eb; border-radius:10px; background:#f8fafc; padding:14px 16px; font-size:0.85rem; line-height:1.35; white-space:pre;">
<code>
--========================================================================
-- GOLD LAYER: MATERIALIZED VIEW FOR BUSINESS ANALYTICS
--========================================================================

CREATE OR REFRESH MATERIALIZED VIEW dq_3_gold.sales_analytics_demo
COMMENT "Business metrics aggregated from validated sales data"
AS
SELECT
  region,
  country,
  category,
  COUNT(DISTINCT order_id) AS total_orders,
  COUNT(DISTINCT customer_id) AS unique_customers,
  SUM(qty) AS total_quantity_sold,
  ROUND(SUM(total_amount), 2) AS total_revenue,
  ROUND(AVG(total_amount), 2) AS avg_order_value,
  ROUND(AVG(discount_pct), 2) AS avg_discount_pct,
  MIN(order_date) AS earliest_order_date,
  MAX(order_date) AS latest_order_date,
  CURRENT_TIMESTAMP() AS last_refreshed_at
FROM dq_2_silver.sales_silver_valid_demo
GROUP BY region, country, category;
</code></pre>

<script>
function copyBlock() {
  const el = document.getElementById("copy-block");
  if (!el) return;

  const text = el.innerText;

  // Preferred modern API
  if (navigator.clipboard && navigator.clipboard.writeText) {
    navigator.clipboard.writeText(text)
      .then(() => alert("Copied to clipboard"))
      .catch(err => {
        console.error("Clipboard write failed:", err);
        fallbackCopy(text);
      });
  } else {
    fallbackCopy(text);
  }
}

function fallbackCopy(text) {
  const textarea = document.createElement("textarea");
  textarea.value = text;
  textarea.style.position = "fixed";
  textarea.style.left = "-9999px";
  document.body.appendChild(textarea);
  textarea.select();
  try {
    document.execCommand("copy");
    alert("Copied to clipboard");
  } finally {
    document.body.removeChild(textarea);
  }
}
</script>

**Review the code:** The gold materialized view pre-computes orders, unique customers, quantity, revenue, average order value, average discount and order date range by region, country and category. It reads only the **valid** silver table, so quarantined records never reach analytics, and it refreshes automatically as new valid data arrives.

## G. Run and Analyze Pipeline Results - First Run (Clean Baseline)

Execute the complete pipeline and analyze the results to validate that the quality framework correctly processes clean baseline data.

### G1. Execute the Complete Pipeline

1. **Run the pipeline** and confirm that it completes successfully with no errors
2. **Observe the pipeline graph** to see data flowing through all layers: Bronze → Silver → Gold
3. **Monitor execution time** and resource utilization in the pipeline UI

**Troubleshooting:** If your pipeline fails, verify that:
- All SQL files are properly created in the `dq_pipeline` folder
- Configuration parameters are correctly set
- Serverless compute is selected and running
- Volume paths are accessible and contain data files

**Expected Pipeline Graph:**

<img src="https://files.training.databricks.com/binder/prod_main/advanced-techniques-with-apache-spark-declarative-pipelines-en_us-1.1.0/images/20261029T211417Z/Advanced Techniques with Apache Spark Declarative Pipelines/Includes/images/dq/checkpoint_run_1.png" alt="Complete Pipeline Run 1" width="1200">

### G2. Validate First Run Results - Clean Baseline

Review the pipeline execution to confirm that all data quality checks passed and the pipeline established a perfect baseline. This run demonstrates that the pipeline framework correctly ingests, transforms, and validates clean data.

**Expected Results for Run 1 (Clean Baseline):**

| Table | Expected Records | Quality Status | Notes |
|-------|------------------|----------------|-------|
| Bronze (sales_bronze_raw_demo) | 157 | All ingested | Raw data from sales_1.csv |
| Silver (sales_bronze_clean_demo) | 157 | Type-cast success | All conversions successful |
| Silver (sales_silver_dq_demo) | 157 | All tracked | All records with is_quarantined = FALSE |
| Silver (sales_silver_valid_demo) | 157 | 100% quality | Perfect baseline established |
| Silver (sales_silver_quarantined_demo) | 0 | No violations | Quality framework validated |
| Gold (sales_analytics_demo) | 55 | Aggregated | Business metrics by region/category |

**Validation Steps in Pipeline UI:**
1. Select the **sales_silver_dq_demo** table and open the **Expectations** tab to verify "6 constraints tracked"
2. Open the **Data** tab to confirm all records have `is_quarantined = FALSE`
3. Check the **sales_silver_quarantined_demo** table to ensure it contains 0 records
4. Verify the **sales_analytics_demo** gold table contains aggregated business metrics

### G3. Query and Analyze Results

Execute the following queries to verify silver and gold layers result:
- Records in **sales_silver_quarantined_demo** should be zero
- Quality Score should be **100%**
- Querying on **sales_analytics_demo** MV for insight

In [ ]:
%sql
-- Verify quarantine is empty for Run 1 (should return 0 - perfect quality baseline)
SELECT 
  COUNT(*) AS quarantined_count,
  'Expected: 0 records for clean baseline' AS validation_note
FROM dq_2_silver.sales_silver_quarantined_demo;

In [ ]:
%sql
-- Verify all records passed quality checks
SELECT 
  COUNT(*) AS total_records,
  SUM(CASE WHEN is_quarantined = FALSE THEN 1 ELSE 0 END) AS valid_records,
  SUM(CASE WHEN is_quarantined = TRUE THEN 1 ELSE 0 END) AS quarantined_records,
  ROUND(SUM(CASE WHEN is_quarantined = FALSE THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS quality_score_pct
FROM dq_2_silver.sales_silver_dq_demo;

In [ ]:
%sql
--Querying on sales_analytics materailized view
SELECT * FROM dq_3_gold.sales_analytics_demo

## H. Prepare and Execute Second Run - Quality Issues and Schema Evolution

Introduce the second data file containing quality issues and new schema columns to demonstrate the pipeline's resilience and monitoring capabilities.

### H1. Land the Second File

Copy the second file (`sales_2.csv`) from the **ops** location to the **sales** location, then list the **sales** volume to confirm both files are there. The pipeline will process the new file incrementally.

In [ ]:
ops_path = f'/Volumes/{my_catalog}/dq_1_bronze/ops'
sales_path = f'/Volumes/{my_catalog}/dq_1_bronze/sales'

# Copy files from ops location to user's source volume (now including sales_2.csv)
copy_files(copy_from=ops_path, copy_to=sales_path, n=2)

# Confirm both files are in the sales volume
spark.sql(f"LIST '{sales_path}'").display()

### H2. Execute Pipeline Run 2 - Quality Issues and Schema Evolution

1. **Run the pipeline again** and confirm that it completes successfully despite quality issues
2. **Observe incremental processing** as the pipeline detects and processes the new file
3. **Monitor quality metrics** in the pipeline UI to see expectation violations being tracked

**Expected Behavior:**
- Pipeline continues running despite quality violations (WARN action)
- New columns are automatically incorporated (schema evolution)
- Invalid records are systematically quarantined with detailed failure reasons
- Valid records continue to flow to analytics tables

**Expected Pipeline Results:**

<img src="https://files.training.databricks.com/binder/prod_main/advanced-techniques-with-apache-spark-declarative-pipelines-en_us-1.1.0/images/20261029T211417Z/Advanced Techniques with Apache Spark Declarative Pipelines/Includes/images/dq/checkpoint_run_2.png" alt="Pipeline Run 2 with Quality Issues" width="1200">

### H3. Analyze Run 2 Results - Quality Issues Detected

Review the pipeline execution to see how quality issues and schema evolution were successfully handled by the quarantine framework.

**Expected Results for Run 2 (Incremental with Issues):**

| Table | New Records | Total Records | Quality Notes |
|-------|-------------|---------------|---------------|
| Bronze (sales_bronze_raw_demo) | +60 | 217 | Schema evolved! (2 new columns populated) |
| Bronze (sales_bronze_clean_demo) | +60 | 217 | Type casting successful for all records |
| Silver (sales_silver_dq_demo) | +60 | 217 | All records tracked with quality flags |
| Silver (sales_silver_valid_demo) | +51 | 208 | 9 records failed expectations |
| Silver (sales_silver_quarantined_demo) | +9 | 9 | Invalid records captured with failure details |
| Gold (sales_analytics_demo) | Updated | 60 | Metrics updated with new valid data |

**Key Validation Points:**
1. **Zero Data Loss**: Bronze (217) = Valid (208) + Quarantined (9)
2. **Schema Evolution**: New columns populated automatically
3. **Quality Detection**: Specific violations identified and quarantined
4. **Pipeline Resilience**: Continues processing despite quality issues

### H4. Analyze the Quality Results

Run the queries below to analyze the quality results of both files:
1. **Overall quality and zero data loss:** total ingested = valid + quarantined.
2. **Quarantined records:** the specific failure reason for each record.
3. **Quality by source file:** the clean baseline file compared with the file that has quality issues.

In [ ]:
%sql
-- Overall quality metrics and zero data loss verification
WITH quality_stats AS (
  SELECT 
    (SELECT COUNT(*) FROM dq_1_bronze.sales_bronze_raw_demo) AS total_ingested,
    (SELECT COUNT(*) FROM dq_2_silver.sales_silver_valid_demo) AS total_valid,
    (SELECT COUNT(*) FROM dq_2_silver.sales_silver_quarantined_demo) AS total_quarantined
)
SELECT 
  total_ingested,
  total_valid,
  total_quarantined,
  ROUND(total_valid * 100.0 / total_ingested, 2) AS quality_score_pct,
  ROUND(total_quarantined * 100.0 / total_ingested, 2) AS failure_rate_pct,
  CASE WHEN total_ingested = total_valid + total_quarantined 
       THEN '✅ ZERO DATA LOSS VERIFIED' 
       ELSE '❌ DATA LOSS DETECTED' 
  END AS data_loss_check
FROM quality_stats;

In [ ]:
%sql
-- Analyze specific quarantined records with detailed failure reasons
SELECT 
  order_id,
  customer_id,
  discount_pct,
  order_date,
  shipping_cost,
  source_file,
  quarantine_reason
FROM dq_2_silver.sales_silver_quarantined_demo
ORDER BY quarantine_reason, order_id;

In [ ]:
%sql
-- Quality score analysis by source file
SELECT 
  COALESCE(b.source_file, q.source_file) AS source_file,
  COUNT(DISTINCT COALESCE(b.order_id, q.order_id)) AS total_records,
  COUNT(DISTINCT s.order_id) AS valid_records,
  COUNT(DISTINCT q.order_id) AS quarantined_records,
  ROUND(COUNT(DISTINCT s.order_id) * 100.0 / 
        NULLIF(COUNT(DISTINCT COALESCE(b.order_id, q.order_id)), 0), 2) AS quality_score_pct,
  CASE 
    WHEN COALESCE(b.source_file, q.source_file) LIKE '%sales_1%' THEN 'Clean baseline data'
    WHEN COALESCE(b.source_file, q.source_file) LIKE '%sales_2%' THEN 'Quality issues + schema evolution'
    ELSE 'Unknown file pattern'
  END AS file_profile
FROM dq_1_bronze.sales_bronze_raw_demo b
FULL OUTER JOIN dq_2_silver.sales_silver_valid_demo s 
  ON b.order_id = s.order_id
FULL OUTER JOIN dq_2_silver.sales_silver_quarantined_demo q 
  ON COALESCE(b.order_id, s.order_id) = q.order_id
GROUP BY COALESCE(b.source_file, q.source_file)
ORDER BY source_file;

**Key observations:**
- **Zero data loss:** bronze (217) = valid (208) + quarantined (9).
- **Specific failure reasons:** each quarantined record lists why it failed (invalid discount, order date outside the 4-year range, invalid shipping cost or missing customer_id), so teams can prioritize remediation.
- **Quality by file:** `sales_1.csv` scores 100%, `sales_2.csv` scores 85% (51 of 60), and the combined score is 95.85% (208 of 217).

### H5. Validate Schema Evolution

Verify that schema evolution worked without any pipeline code changes. Records from the first file have NULL values for the new columns (**shipping_cost** and **order_status**), while records from the second file have populated values.

In [ ]:
%sql
SELECT *
FROM dq_2_silver.sales_silver_valid_demo

**What this shows:**
- The schema grew from 16 to 18 business columns without any pipeline code changes, because the schema hints in the bronze table already declared the new columns.
- Records from `sales_1.csv` keep NULL in the new columns, and records from `sales_2.csv` populate them.
- The new `shipping_cost` column is automatically included in the quality checks (`valid_shipping_cost`).

## I. Summary and Key Takeaways

**Data Quality Management:**
- Successfully implemented 6 comprehensive data quality expectations covering business keys, numeric ranges, and temporal validations
- Achieved 100% quality score on clean baseline data (Run 1: 157/157 records)
- Detected and quarantined 9 quality violations in problematic data (Run 2: 51/60 records passed)
- Overall pipeline quality score: 95.85% (208/217 total records processed)

**Schema Evolution:**
- Seamlessly handled schema evolution from 16 to 18 core business columns from source without pipeline code changes
- Bronze layer STRING strategy enabled backward compatibility and forward flexibility
- New business columns (**order_status, shipping_cost**) automatically integrated

**Zero Data Loss Quarantine:**
- Implemented official Databricks inverse logic pattern for quarantine processing
- Achieved mathematical zero data loss: Bronze Records = Valid Sales Records + Quarantine Sales Records
- Provided detailed failure analysis and remediation insights for each quarantined record

### Production Readiness

This demonstration showcased enterprise-grade patterns that provide:
- **Robust data quality controls** that catch anomalies before they reach analytics
- **Flexible schema handling** that adapts to evolving business requirements
- **Complete data lineage** with zero loss and full audit capabilities
- **Actionable quality insights** for continuous data improvement

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>